In [ ]:
#Install Packages
%pip install -U langgraph langchain langchain-core langchain-community langchain-groq faiss-cpu sentence-transformers python-dotenv pandas

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
# Restart Python
dbutils.library.restartPython()

In [ ]:
# Imports
import os
import json
import math
import pandas as pd
import numpy as np

from typing import TypedDict, Optional, List, Dict, Any

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

import faiss
from sentence_transformers import SentenceTransformer

/local_disk0/.ephemeral_nfs/envs/pythonEnv-3ff3dc4b-afb8-41be-bad2-5acf6d337cbf/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


In [ ]:
# API key setup
os.environ["GROQ_API_KEY"] = "REPLACE_WITH_YOUR_GROQ_API_KEY"
print("Key loaded:", os.getenv("GROQ_API_KEY") is not None)

Key loaded: True


In [ ]:
# Load the real tables
gold_features_spark = spark.table("workspace.default.gold_features")
gold_nlp_spark      = spark.table("workspace.default.gold_nlp_features")

In [ ]:
print("gold_features unique listings:",
      gold_features_spark.select("listing_id").distinct().count())

print("gold_nlp unique listings:",
      gold_nlp_spark.select("listing_id").distinct().count())

gold_features unique listings: 33065
gold_nlp unique listings: 35217


In [ ]:
print("gold_features total rows:", gold_features_spark.count())
print("gold_nlp total rows:", gold_nlp_spark.count())

gold_features total rows: 1955039
gold_nlp total rows: 35217


In [ ]:
# Convert to pandas for the agent layer
features_df = gold_features_spark.toPandas()
nlp_df      = gold_nlp_spark.toPandas()

In [ ]:
#Join tables
df = features_df.merge(
    nlp_df,
    on=["listing_id", "city"],
    how="left"
)
df.display()

,listing_id,city,neighborhood,room_type,price,median_neighborhood_price,price_gap_pct,occupancy_rate,occupancy_rank_percentile,amenity_score,review_rank_percentile,competitive_score,peak_month,peak_occupancy_rate,avg_sentiment_score,pct_positive,total_reviews,cleanliness_mentions,wifi_mentions,checkin_mentions,location_mentions,noise_mentions,host_mentions,value_mentions,sentiment_category,top_praise,top_complaint
0,736997706091137832,austin,78701,entire home/apt,75.0,197.0,-61.93,0.0,0.0,51.02,0.00,42.76,8,0.3655,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1050810576575216662,austin,78701,entire home/apt,189.0,197.0,-4.06,0.0,0.0,33.67,0.00,24.64,8,0.3655,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1147691461254312780,austin,78701,entire home/apt,380.0,197.0,92.89,0.0,0.0,34.69,0.00,8.67,8,0.3655,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1147692298498919319,austin,78701,entire home/apt,260.0,197.0,31.98,0.0,0.0,34.69,0.00,14.08,8,0.3655,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1284017961958369153,austin,78701,private room,110.0,179.0,-38.55,0.0,0.0,32.65,0.00,34.73,8,0.3655,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1955034,874573771525421667,seattle,Yesler Terrace,entire home/apt,124.0,123.0,0.81,1.0,99.2,69.64,87.84,73.97,9,0.3206,0.772,0.963,190.0,57.0,0.0,14.0,76.0,18.0,57.0,4.0,Very Positive,location,cleanliness
1955035,868968319858852604,seattle,Yesler Terrace,entire home/apt,112.0,123.0,-8.94,1.0,99.2,67.86,90.26,77.05,9,0.3206,0.772,0.940,199.0,54.0,1.0,21.0,92.0,13.0,79.0,7.0,Very Positive,location,cleanliness
1955036,42628302,seattle,Yesler Terrace,entire home/apt,85.0,123.0,-30.89,1.0,99.2,60.71,92.82,82.49,9,0.3206,0.670,0.904,228.0,39.0,4.0,24.0,133.0,31.0,63.0,17.0,Very Positive,location,cleanliness
1955037,27434633,seattle,Yesler Terrace,entire home/apt,97.0,123.0,-21.14,1.0,99.2,57.14,95.23,79.27,9,0.3206,0.755,0.940,301.0,75.0,3.0,35.0,161.0,35.0,117.0,32.0,Very Positive,location,cleanliness


In [ ]:
# Fill NLP nulls so the prompt stay stable
fill_values = {
    "avg_sentiment_score": 0.0,
    "sentiment_category": "Unknown",
    "pct_positive": 0.0,
    "total_reviews": 0,
    "top_complaint": "unknown",
    "top_praise": "unknown",
    "cleanliness_mentions": 0,
    "wifi_mentions": 0,
    "checkin_mentions": 0,
    "location_mentions": 0,
    "noise_mentions": 0,
    "host_mentions": 0,
    "value_mentions": 0
}

for col, val in fill_values.items():
    if col in df.columns:
        df[col] = df[col].fillna(val)

df.display()

,listing_id,city,neighborhood,room_type,price,median_neighborhood_price,price_gap_pct,occupancy_rate,occupancy_rank_percentile,amenity_score,review_rank_percentile,competitive_score,peak_month,peak_occupancy_rate,avg_sentiment_score,pct_positive,total_reviews,cleanliness_mentions,wifi_mentions,checkin_mentions,location_mentions,noise_mentions,host_mentions,value_mentions,sentiment_category,top_praise,top_complaint
0,736997706091137832,austin,78701,entire home/apt,75.0,197.0,-61.93,0.0,0.0,51.02,0.00,42.76,8,0.3655,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,unknown,unknown
1,1050810576575216662,austin,78701,entire home/apt,189.0,197.0,-4.06,0.0,0.0,33.67,0.00,24.64,8,0.3655,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,unknown,unknown
2,1147691461254312780,austin,78701,entire home/apt,380.0,197.0,92.89,0.0,0.0,34.69,0.00,8.67,8,0.3655,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,unknown,unknown
3,1147692298498919319,austin,78701,entire home/apt,260.0,197.0,31.98,0.0,0.0,34.69,0.00,14.08,8,0.3655,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,unknown,unknown
4,1284017961958369153,austin,78701,private room,110.0,179.0,-38.55,0.0,0.0,32.65,0.00,34.73,8,0.3655,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,unknown,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1955034,874573771525421667,seattle,Yesler Terrace,entire home/apt,124.0,123.0,0.81,1.0,99.2,69.64,87.84,73.97,9,0.3206,0.772,0.963,190.0,57.0,0.0,14.0,76.0,18.0,57.0,4.0,Very Positive,location,cleanliness
1955035,868968319858852604,seattle,Yesler Terrace,entire home/apt,112.0,123.0,-8.94,1.0,99.2,67.86,90.26,77.05,9,0.3206,0.772,0.940,199.0,54.0,1.0,21.0,92.0,13.0,79.0,7.0,Very Positive,location,cleanliness
1955036,42628302,seattle,Yesler Terrace,entire home/apt,85.0,123.0,-30.89,1.0,99.2,60.71,92.82,82.49,9,0.3206,0.670,0.904,228.0,39.0,4.0,24.0,133.0,31.0,63.0,17.0,Very Positive,location,cleanliness
1955037,27434633,seattle,Yesler Terrace,entire home/apt,97.0,123.0,-21.14,1.0,99.2,57.14,95.23,79.27,9,0.3206,0.755,0.940,301.0,75.0,3.0,35.0,161.0,35.0,117.0,32.0,Very Positive,location,cleanliness


In [ ]:
# Ensuring listing_id is string for clean lookup
df["listing_id"] = df["listing_id"].astype(str)

In [ ]:
# Add business logic helpers
def classify_price_position(price_gap_pct: float) -> str:
    if pd.isna(price_gap_pct):
        return "unknown"
    if price_gap_pct <= -20:
        return "significantly underpriced"
    elif price_gap_pct <= -5:
        return "slightly underpriced"
    elif price_gap_pct < 5:
        return "roughly at market"
    elif price_gap_pct < 20:
        return "slightly overpriced"
    else:
        return "significantly overpriced"


def classify_competitive_tier(score: float) -> str:
    if pd.isna(score):
        return "unknown"
    if score > 80:
        return "Market Leader"
    elif score >= 40:
        return "Mid-tier"
    return "Fixer-Upper"


def classify_sentiment(score: float) -> str:
    if pd.isna(score):
        return "unknown"
    if score >= 0.5:
        return "very positive"
    elif score >= 0.15:
        return "positive"
    elif score > -0.15:
        return "neutral"
    elif score > -0.5:
        return "negative"
    return "very negative"

In [ ]:
#  Build FAISS comparables
compare_cols = [
    "listing_id", "city", "neighborhood", "room_type", "price",
    "median_neighborhood_price", "price_gap_pct", "occupancy_rate",
    "occupancy_rank_percentile", "amenity_score", "review_rank_percentile",
    "competitive_score", "top_complaint", "top_praise", "avg_sentiment_score"
]

for col in compare_cols:
    if col not in df.columns:
        df[col] = np.nan

In [ ]:
# Create searchable text
def build_listing_text(row):
    return (
        f"Listing {row['listing_id']} in {row['neighborhood']}, {row['city']}. "
        f"Room type: {row['room_type']}. "
        f"Price: {row['price']}. "
        f"Neighborhood median: {row['median_neighborhood_price']}. "
        f"Price gap percent: {row['price_gap_pct']}. "
        f"Occupancy rate: {row['occupancy_rate']}. "
        f"Occupancy percentile: {row['occupancy_rank_percentile']}. "
        f"Amenity score: {row['amenity_score']}. "
        f"Review percentile: {row['review_rank_percentile']}. "
        f"Competitive score: {row['competitive_score']}. "
        f"Top complaint: {row['top_complaint']}. "
        f"Top praise: {row['top_praise']}. "
        f"Average sentiment: {row['avg_sentiment_score']}."
    )

df["listing_text"] = df.apply(build_listing_text, axis=1)

In [ ]:
#Check the Length of Listing text
texts = df["listing_text"].fillna("").astype(str).tolist()
print("Total rows:", len(texts))

Total rows: 1955039


In [ ]:
embed_df = df.drop_duplicates(subset=["listing_id"]).copy()
print("Original rows:", len(df))
print("Unique listings:", embed_df["listing_id"].nunique())
texts = embed_df["listing_text"].fillna("").astype(str).tolist()

Original rows: 1955039
Unique listings: 33065


In [ ]:
# Load embedding model and build FAISS index
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # cosine-like since normalized
index.add(embeddings.astype("float32"))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/517 [00:00<?, ?it/s]

In [ ]:
# Map row positions
embed_df = embed_df.reset_index(drop=True).copy()
embed_df["listing_id"] = embed_df["listing_id"].astype(str)

# Map row positions based on embed_df
row_id_map = {i: embed_df.iloc[i]["listing_id"] for i in range(len(embed_df))}
listing_to_row = {embed_df.iloc[i]["listing_id"]: i for i in range(len(embed_df))}

In [ ]:
# Retriever for comparables
def get_comparable_listings(listing_id: str, top_k: int = 4) -> pd.DataFrame:
    if listing_id not in listing_to_row:
        return pd.DataFrame()

    idx = listing_to_row[listing_id]
    query_vec = embeddings[idx].reshape(1, -1).astype("float32")
    scores, neighbors = index.search(query_vec, top_k + 1)

    neighbor_ids = []
    for n_idx in neighbors[0]:
        candidate_id = row_id_map[n_idx]
        if candidate_id != listing_id:
            neighbor_ids.append(candidate_id)

    if not neighbor_ids:
        return pd.DataFrame()

    comp_df = embed_df[embed_df["listing_id"].isin(neighbor_ids)].copy()

    target = embed_df[embed_df["listing_id"] == listing_id].iloc[0]

    # Prefer same city / neighborhood / room type when possible
    comp_df["same_city"] = (comp_df["city"] == target["city"]).astype(int)
    comp_df["same_neighborhood"] = (comp_df["neighborhood"] == target["neighborhood"]).astype(int)
    comp_df["same_room_type"] = (comp_df["room_type"] == target["room_type"]).astype(int)

    comp_df = comp_df.sort_values(
        by=["same_neighborhood", "same_room_type", "same_city", "competitive_score"],
        ascending=[False, False, False, False]
    ).head(3)

    return comp_df

In [ ]:
def retrieve_listing_context(listing_id: str) -> str:
    listing_id = str(listing_id)
    row_df = embed_df[embed_df["listing_id"] == listing_id]

    if row_df.empty:
        return f"Listing {listing_id} was not found."

    row = row_df.iloc[0]

    price_position = classify_price_position(row["price_gap_pct"])
    market_tier = classify_competitive_tier(row["competitive_score"])
    sentiment_label = classify_sentiment(row["avg_sentiment_score"])

    comps = get_comparable_listings(listing_id, top_k=4)

    context = f"""
LISTING FACTS
Listing ID: {row['listing_id']}
City: {row['city']}
Neighborhood: {row['neighborhood']}
Room Type: {row['room_type']}
Current Price: ${row['price']:.2f}
Neighborhood Median Price: ${row['median_neighborhood_price']:.2f}
Price Gap Percent: {row['price_gap_pct']:.2f}%
Price Position: {price_position}
Occupancy Rate (90d): {row['occupancy_rate']:.4f}
Occupancy Rank Percentile: {row['occupancy_rank_percentile']:.2f}
Amenity Score: {row['amenity_score']:.2f}
Review Rank Percentile: {row['review_rank_percentile']:.2f}
Competitive Score: {row['competitive_score']:.2f}
Competitive Tier: {market_tier}
Peak Month: {int(row['peak_month']) if not pd.isna(row['peak_month']) else 0}
Peak Occupancy Rate: {0.0 if pd.isna(row['peak_occupancy_rate']) else row['peak_occupancy_rate']:.4f}

NLP FACTS
Average Sentiment Score: {row['avg_sentiment_score']:.4f}
Sentiment Label: {sentiment_label}
Sentiment Category Table Label: {row.get('sentiment_category', 'Unknown')}
Percent Positive Reviews: {row['pct_positive']:.4f}
Total Reviews: {int(row['total_reviews'])}
Top Complaint: {row['top_complaint']}
Top Praise: {row['top_praise']}
Cleanliness Mentions: {int(row['cleanliness_mentions'])}
Wifi Mentions: {int(row['wifi_mentions'])}
Check-in Mentions: {int(row['checkin_mentions'])}
Location Mentions: {int(row['location_mentions'])}
Noise Mentions: {int(row['noise_mentions'])}
Host Mentions: {int(row['host_mentions'])}
Value Mentions: {int(row['value_mentions'])}
""".strip()

    if not comps.empty:
        comp_lines = []
        for _, c in comps.iterrows():
            comp_lines.append(
                f"- Listing {c['listing_id']} | {c['neighborhood']}, {c['city']} | "
                f"{c['room_type']} | Price ${c['price']:.2f} | "
                f"Price Gap {c['price_gap_pct']:.2f}% | "
                f"Occupancy {c['occupancy_rate']:.4f} | "
                f"Competitive Score {c['competitive_score']:.2f} | "
                f"Top Complaint: {c['top_complaint']} | Top Praise: {c['top_praise']}"
            )
        context += "\n\nCOMPARABLE LISTINGS\n" + "\n".join(comp_lines)

    return context

In [ ]:
# Test of the retriever context
test_ids = embed_df["listing_id"].astype(str).head(3).tolist()
for lid in test_ids:
    print("=" * 80)
    print(retrieve_listing_context(lid))

LISTING FACTS
Listing ID: 736997706091137832
City: austin
Neighborhood: 78701
Room Type: entire home/apt
Current Price: $75.00
Neighborhood Median Price: $197.00
Price Gap Percent: -61.93%
Price Position: significantly underpriced
Occupancy Rate (90d): 0.0000
Occupancy Rank Percentile: 0.00
Amenity Score: 51.02
Review Rank Percentile: 0.00
Competitive Score: 42.76
Competitive Tier: Mid-tier
Peak Month: 8
Peak Occupancy Rate: 0.3655

NLP FACTS
Average Sentiment Score: 0.0000
Sentiment Label: neutral
Sentiment Category Table Label: Unknown
Percent Positive Reviews: 0.0000
Total Reviews: 0
Top Complaint: unknown
Top Praise: unknown
Cleanliness Mentions: 0
Wifi Mentions: 0
Check-in Mentions: 0
Location Mentions: 0
Noise Mentions: 0
Host Mentions: 0
Value Mentions: 0

COMPARABLE LISTINGS
- Listing 681607732520899659 | 78701, austin | entire home/apt | Price $212.00 | Price Gap 7.61% | Occupancy 0.0000 | Competitive Score 28.79 | Top Complaint: unknown | Top Praise: unknown
- Listing 7499727

Building the Prompt

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [ ]:

analysis_prompt = ChatPromptTemplate.from_messages([

    ("system", """
You are an Airbnb market analyst.
Your job is to analyze ONE listing using ONLY the numbers and facts in the provided context.

Rules:
1. Use only information explicitly present in the context.

2. Do not invent any statistics, percentages, ranks, or prices.

3. If a field is unknown or missing, say it is unknown.

4. Be concise and structured.

5. Focus on:
   - pricing position
   - demand/occupancy position
   - competitiveness
   - guest sentiment
   - most important issue and most important strength

Return exactly this format:

PRICING POSITION:
- ...

DEMAND POSITION:
- ...

COMPETITIVE POSITION:
- ...

GUEST FEEDBACK SUMMARY:
- ...

TOP STRENGTH:
- ...

TOP ISSUE:
- ...

"""),

    ("human", "Context:\n{context}")

])

In [ ]:
#Recommendation Prompt
recommend_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an Airbnb strategy advisor.

Using ONLY the provided context and analysis, produce exactly 3 specific, actionable recommendations.

Rules:
1. Use ONLY facts and numbers that appear in the context or analysis.
2. Never invent numbers.
3. Every recommendation must cite the source stat(s) it uses.
4. Recommendations must be practical for a host.
5. Keep each recommendation to 2-4 sentences.
6. If the listing is underpriced with strong occupancy, you may recommend a price increase test.
7. If the listing is overpriced with weak occupancy, you may recommend a price reduction test.
8. If sentiment/top complaint suggests operational problems, include an operational fix.
9. Reference comparable listings when helpful.
10. Output exactly 3 numbered recommendations.

Return format:

1. Recommendation title
Recommendation text
Source stats: ...

2. Recommendation title
Recommendation text
Source stats: ...

3. Recommendation title
Recommendation text
Source stats: ...
"""),
    ("human", "Context:\n{context}\n\nAnalysis:\n{analysis}")
])

In [ ]:
#Building the LangGraph flow
class AgentState(TypedDict):
    listing_id: str
    context: str
    analysis: str
    action_plan: str

In [ ]:
#Retriever Node
def retrieve_node(state: AgentState) -> AgentState:
    listing_id = str(state["listing_id"])
    context = retrieve_listing_context(listing_id)
    return {
        **state,
        "context": context
    }

In [ ]:
#Analysis Node
def analyze_node(state: AgentState) -> AgentState:
    prompt = analysis_prompt.format_messages(context=state["context"])
    response = llm.invoke(prompt)
    analysis = response.content if hasattr(response, "content") else str(response)
    return {
        **state,
        "analysis": analysis
    }

In [ ]:
#Recommend Node
def recommend_node(state: AgentState) -> AgentState:
    prompt = recommend_prompt.format_messages(
        context=state["context"],
        analysis=state["analysis"]
    )
    response = llm.invoke(prompt)
    action_plan = response.content if hasattr(response, "content") else str(response)
    return {
        **state,
        "action_plan": action_plan
    }

In [ ]:
#Graph of Agent State
graph = StateGraph(AgentState)

graph.add_node("retrieve", retrieve_node)
graph.add_node("analyze", analyze_node)
graph.add_node("recommend", recommend_node)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "analyze")
graph.add_edge("analyze", "recommend")
graph.add_edge("recommend", END)

app = graph.compile()

In [ ]:
#Test of Agent
sample_id = df["listing_id"].iloc[0]
result = app.invoke({"listing_id": str(sample_id), "context": "", "analysis": "", "action_plan": ""})

print("CONTEXT\n", result["context"])
print("\nANALYSIS\n", result["analysis"])
print("\nACTION PLAN\n", result["action_plan"])

CONTEXT
 LISTING FACTS
Listing ID: 736997706091137832
City: austin
Neighborhood: 78701
Room Type: entire home/apt
Current Price: $75.00
Neighborhood Median Price: $197.00
Price Gap Percent: -61.93%
Price Position: significantly underpriced
Occupancy Rate (90d): 0.0000
Occupancy Rank Percentile: 0.00
Amenity Score: 51.02
Review Rank Percentile: 0.00
Competitive Score: 42.76
Competitive Tier: Mid-tier
Peak Month: 8
Peak Occupancy Rate: 0.3655

NLP FACTS
Average Sentiment Score: 0.0000
Sentiment Label: neutral
Sentiment Category Table Label: Unknown
Percent Positive Reviews: 0.0000
Total Reviews: 0
Top Complaint: unknown
Top Praise: unknown
Cleanliness Mentions: 0
Wifi Mentions: 0
Check-in Mentions: 0
Location Mentions: 0
Noise Mentions: 0
Host Mentions: 0
Value Mentions: 0

COMPARABLE LISTINGS
- Listing 681607732520899659 | 78701, austin | entire home/apt | Price $212.00 | Price Gap 7.61% | Occupancy 0.0000 | Competitive Score 28.79 | Top Complaint: unknown | Top Praise: unknown
- Listin

In [ ]:
# a clean callable function for Streamlit
def action_plan(listing_id: str) -> str:
    result = app.invoke({
        "listing_id": str(listing_id),
        "context": "",
        "analysis": "",
        "action_plan": ""
    })
    return result["action_plan"]

In [ ]:
#Distinct action Plan function
print(action_plan(df["listing_id"].iloc[5]))

1. Price Reduction Test
The listing is significantly overpriced with a price gap percent of 41.12% compared to the neighborhood median price of $197.00. Reducing the price to be more competitive with the neighborhood median price may help increase occupancy. 
Source stats: Price Gap Percent: 41.12%, Neighborhood Median Price: $197.00

2. Competitive Improvement
The listing has a competitive score of 19.25, which places it in the "Fixer-Upper" competitive tier, indicating a need for improvement to be competitive. Improving the listing's amenities and overall guest experience may help increase its competitiveness and attract more bookings.
Source stats: Competitive Score: 19.25, Competitive Tier: Fixer-Upper

3. Review Collection and Analysis
The listing has no reviews, resulting in a neutral sentiment label and an average sentiment score of 0.0000. Collecting reviews from guests and analyzing them to identify areas for improvement may help the host address any operational issues and inc

In [ ]:
#Add a validation check for hallucinations
#Extract numbers from text
import re

def extract_numbers(text: str):
    if not text:
        return set()
    matches = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return set(matches)

In [ ]:
# Validation function
def validate_output_grounding(context: str, action_plan: str) -> bool:
    context_numbers = extract_numbers(context)
    output_numbers = extract_numbers(action_plan)

    # allow numbering 1,2,3 in recommendations
    output_numbers = {n for n in output_numbers if n not in {"1", "2", "3"}}

    return output_numbers.issubset(context_numbers)

In [ ]:
# Run on 10 listings
validation_rows = []

sample_ids = df["listing_id"].astype(str).sample(
    n=min(10, len(df)),
    random_state=42
).tolist()

for lid in sample_ids:
    result = app.invoke({
        "listing_id": lid,
        "context": "",
        "analysis": "",
        "action_plan": ""
    })

    passed = validate_output_grounding(result["context"], result["action_plan"])

    validation_rows.append({
        "listing_id": lid,
        "pass_grounding_check": passed
    })

validation_log = pd.DataFrame(validation_rows)
validation_log

,listing_id,pass_grounding_check
0,714657251637718261,True
1,1418388273455451419,True
2,985694521248486231,True
3,11257096,True
4,16823279,True
5,1320022753016805471,False
6,3783533,True
7,52267319,True
8,17655474,True
9,10332096,True


In [ ]:
# Summary of Validation
print(validation_log["pass_grounding_check"].value_counts(dropna=False))

pass_grounding_check
True     9
False    1
Name: count, dtype: int64


In [ ]:
#If a listing fails, print both fields
failed_ids = validation_log.loc[
    validation_log["pass_grounding_check"] == False, "listing_id"
].tolist()

for lid in failed_ids:
    result = app.invoke({
        "listing_id": lid,
        "context": "",
        "analysis": "",
        "action_plan": ""
    })
    print("="*80)
    print("LISTING:", lid)
    print("CONTEXT:\n", result["context"])
    print("\nACTION PLAN:\n", result["action_plan"])

LISTING: 1320022753016805471
CONTEXT:
 LISTING FACTS
Listing ID: 1320022753016805471
City: seattle
Neighborhood: Dunlap
Room Type: entire home/apt
Current Price: $128.00
Neighborhood Median Price: $141.00
Price Gap Percent: -9.22%
Price Position: slightly underpriced
Occupancy Rate (90d): 0.0000
Occupancy Rank Percentile: 0.00
Amenity Score: 61.64
Review Rank Percentile: 43.19
Competitive Score: 43.98
Competitive Tier: Mid-tier
Peak Month: 8
Peak Occupancy Rate: 0.4291

NLP FACTS
Average Sentiment Score: 0.7260
Sentiment Label: very positive
Sentiment Category Table Label: Very Positive
Percent Positive Reviews: 0.8890
Total Reviews: 18
Top Complaint: cleanliness
Top Praise: cleanliness
Cleanliness Mentions: 10
Wifi Mentions: 1
Check-in Mentions: 2
Location Mentions: 6
Noise Mentions: 3
Host Mentions: 7
Value Mentions: 2

COMPARABLE LISTINGS
- Listing 656531036034625298 | Dunlap, seattle | entire home/apt | Price $113.00 | Price Gap -19.86% | Occupancy 0.0000 | Competitive Score 38.75 